# Domain Adaptation

## Loading the Datasets

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Define transformation pipeline for MNIST
transform_mnist = transforms.Compose([
    transforms.Resize(32),             # match SVHN size
    transforms.Grayscale(3),           # convert to 3 channels (because SVHN images are not grayscale)
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Define transformation pipeline for SVHN
transform_svhn = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Get train and test sets for MNIST
mnist_train = datasets.MNIST(root="./data", train=True, download=True, transform=transform_mnist)
mnist_test  = datasets.MNIST(root="./data", train=False, transform=transform_mnist)

# Get train and test sets for SVHN
svhn_train = datasets.SVHN(root="./data", split="train", download=True, transform=transform_svhn)
svhn_test  = datasets.SVHN(root="./data", split="test", download=True, transform=transform_svhn)

# Define data loaders
mnist_loader = DataLoader(mnist_train, batch_size=128, shuffle=True)
mnist_test_loader = DataLoader(mnist_test, batch_size=128)

svhn_loader = DataLoader(svhn_train, batch_size=128, shuffle=True)
svhn_test_loader = DataLoader(svhn_test, batch_size=128)

## Model Definition

In [5]:
# Define a simple CNN for feature extraction
class FeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 32, 5),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 48, 5),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten()
        )

    def forward(self, x):
        return self.net(x)

# Define a final MLP for classification
class Classifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(48 * 5 * 5, 100),
            nn.ReLU(),
            nn.Linear(100, 10)
        )

    def forward(self, x):
        return self.fc(x)

# Instantiate models
feature_extractor = FeatureExtractor().to(device)
classifier = Classifier().to(device)


Training on MNIST...
Epoch 1 done
Epoch 2 done
Epoch 3 done
Epoch 4 done
Epoch 5 done

Before adaptation:
MNIST accuracy: 0.9912
SVHN accuracy: 0.3368

Domain adaptation...
DA Epoch 1 done | Loss: 0.0205
DA Epoch 2 done | Loss: 0.0163
DA Epoch 3 done | Loss: 0.0145
DA Epoch 4 done | Loss: 0.0095
DA Epoch 5 done | Loss: 0.0094

After adaptation:
MNIST accuracy: 0.9932
SVHN accuracy: 0.3460


0.34603564843269824

## Training Loop

In [ ]:
# Define classification loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(feature_extractor.parameters()) + list(classifier.parameters()), lr=1e-3)

# Define standard training loop to train the model over mnist
def train_mnist(epochs=5):
    feature_extractor.train()
    classifier.train()

    for epoch in range(epochs):
        for x, y in mnist_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            features = feature_extractor(x)
            preds = classifier(features)

            loss = criterion(preds, y)
            loss.backward()
            optimizer.step()

        print(f"Epoch {epoch+1} done")

# Define evaluation function
def evaluate(loader, name="dataset"):
    feature_extractor.eval()
    classifier.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = classifier(feature_extractor(x))
            preds = preds.argmax(dim=1)

            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = correct / total
    print(f"{name} accuracy: {acc:.4f}")
    return acc

## Domain Adaptation Training

In [ ]:
# Define MMD Loss function
def gaussian_kernel(x, y, sigma=1.0):
    x = x.unsqueeze(1)
    y = y.unsqueeze(0)
    return torch.exp(-((x - y) ** 2).sum(2) / (2 * sigma ** 2))

def mmd_loss(x, y):
    Kxx = gaussian_kernel(x, x)
    Kyy = gaussian_kernel(y, y)
    Kxy = gaussian_kernel(x, y)
    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()


# Define training for Domain Adaptation
def train_domain_adaptation(epochs=5, lambda_mmd=0.5):
    feature_extractor.train()
    classifier.train()

    svhn_iter = iter(svhn_loader)

    for epoch in range(epochs):
        for mnist_x, mnist_y in mnist_loader:

            try:
                svhn_x, _ = next(svhn_iter)
            except StopIteration:
                svhn_iter = iter(svhn_loader)
                svhn_x, _ = next(svhn_iter)

            mnist_x, mnist_y = mnist_x.to(device), mnist_y.to(device)
            svhn_x = svhn_x.to(device)

            optimizer.zero_grad()

            # source (MNIST)
            f_src = feature_extractor(mnist_x)
            preds = classifier(f_src)
            cls_loss = criterion(preds, mnist_y)

            # target (SVHN)
            f_tgt = feature_extractor(svhn_x)

            # MMD
            mmd = mmd_loss(f_src, f_tgt)

            loss = cls_loss + lambda_mmd * mmd
            loss.backward()
            optimizer.step()

        print(f"DA Epoch {epoch+1} done | Loss: {loss.item():.4f}")

## Final Pipeline

In [ ]:
# Run entire pipeline
print("\nTraining on MNIST...")
train_mnist()

print("\nBefore adaptation:")
evaluate(mnist_test_loader, "MNIST")
evaluate(svhn_test_loader, "SVHN")

print("\nDomain adaptation...")
train_domain_adaptation()

print("\nAfter adaptation:")
evaluate(mnist_test_loader, "MNIST")
evaluate(svhn_test_loader, "SVHN")